# Make ML models for prediction gte parameters

In [1]:
%pip install --upgrade openpyxl pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


# Parameters

In [18]:
from config import parameters as gtep

In [19]:
for alias, parameter in gtep.items():
    print(f"{alias:<15}: {parameter}")

l              : thermal_conductivity
hc             : heat_capacity
Cp             : heat_capacity_pressure
Cv             : heat_capacity_volume
k              : adiabatic_index
gc             : gas_const
eo             : excess_oxidizing
T              : static_temperature
P              : static_pressure
D              : staticdensity
TT             : total_temperature
PP             : total_pressure
DD             : total_density
m              : mass
v              : volume
a              : sound_speed
a_critical     : critical_sound_speed
c              : absolute_velocity
u              : portable_velocity
w              : relative_velocity
mf             : mass_flow
pipi           : total_pressure_ratio
pi             : static_pressure_ratio
titi           : total_temperature_ratio
ti             : static_temperature_ratio
mach           : mach_number
Nu             : nusselt_number
efficiency     : efficiency
effeff         : total_efficiency
peff           : pressure_efficie

# Functions

In [20]:
import time
import pickle

In [21]:
def now() -> str:
    return time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())

In [22]:
def dump(path: str, body):
    with open(path, "wb") as file:
        pickle.dump(body, file)


def load(path: str):
    with open(path, "rb") as file:
        return pickle.load(file)

# Generate data

In [23]:
from itertools import product
import numpy as np
import pandas as pd
from tqdm import tqdm


from thermodynamics import gas_const, heat_capacity_p
from substance import Substance

In [24]:
def generate(**kwargs):
    """Генерация"""
    static, names, values = {}, [], []
    for k, v in kwargs.items():
        if isinstance(v, (tuple, list, np.ndarray)):
            names.append(k)
            values.append(v)
        else:
            static[k] = v

    for combination in product(*values):
        result = dict(zip(names, combination))
        yield {**static, **result}

In [ ]:
discretese = 10

substance_parameters = [
    *generate(
        **{
            gtep.mf: np.linspace(1, 200, discretese, endpoint=True),
            gtep.TT: np.linspace(250, 1050, discretese, endpoint=True),
            gtep.PP: np.linspace(1, 20, discretese, endpoint=True) * 10**5,
        }
    )
]

dump(f"data/substance_parameters_{len(substance_parameters)}.pkl.xz", substance_parameters)

len(substance_parameters)

1000

In [47]:
substance_parameters = load(f"data/substance_parameters_{len(substance_parameters)}.pkl.xz")

len(substance_parameters)

1000

In [48]:
functions = {
    gtep.gc: lambda temperature: gas_const("air"),
    gtep.Cp: lambda temperature: heat_capacity_p("air", temperature),
}

## Air

In [49]:
def generate_air(**kwargs):
    """Генерация воздуха"""

    for combination in generate(**kwargs):
        yield Substance(
            combination["name"],
            composition=combination["composition"],
            parameters=combination["parameters"],
            functions=combination["functions"],
        )

In [50]:
airs = [
    *generate_air(
        name="air",
        composition=({"N2": 0.78, "O2": 0.21, "Ar": 0.009, "CO2": 0.0004}),
        parameters=substance_parameters,
        functions=functions,
    )
]

# with open(f"data/airs_{len(substance_parameters)}.pkl", "wb") as file:     pickle.dump(airs, file)

len(airs)

1000

## Compressor

In [51]:
from gte import Compressor

In [54]:
data = []
for air in tqdm(airs):
    for compressor in Compressor.generator(
        **{
            gtep.pipi: np.arange(1.1, 15.1 + 1, 1),
            gtep.effeff: np.arange(0.84, 0.94 + 0.01, 0.01),
        },
    ):
        compressor.solve(air)

        if compressor.is_real:
            data.append(compressor.summary)

dump(f"data/compressor_{len(substance_parameters)}.pkl.xz", data)

100%|██████████| 1000/1000 [01:14<00:00, 13.40it/s]


In [55]:
df = pd.DataFrame(data)
df

,name,inlet_mass_flow,inlet_total_temperature,inlet_total_pressure,outlet_mass_flow,outlet_total_temperature,outlet_total_pressure,outlet_gas_const,outlet_heat_capacity_pressure,outlet_total_density,outlet_adiabatic_index,outlet_critical_sound_speed,leak,total_pressure_ratio,total_efficiency,power
0,Compressor,1.0,250.0,100000.0,1.0,258.243268,110000.0,287.14,1001.731420,1.483440,1.401824,294.206665,0,1.1,0.84,8.257334e+03
1,Compressor,1.0,250.0,100000.0,1.0,258.146291,110000.0,287.14,1001.730502,1.483997,1.401825,294.151441,0,1.1,0.85,8.160189e+03
2,Compressor,1.0,250.0,100000.0,1.0,258.051569,110000.0,287.14,1001.729617,1.484542,1.401825,294.097491,0,1.1,0.86,8.065303e+03
3,Compressor,1.0,250.0,100000.0,1.0,257.959024,110000.0,287.14,1001.728763,1.485075,1.401826,294.044772,0,1.1,0.87,7.972598e+03
4,Compressor,1.0,250.0,100000.0,1.0,257.868583,110000.0,287.14,1001.727939,1.485595,1.401826,293.993241,0,1.1,0.88,7.882001e+03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175995,Compressor,200.0,1050.0,2000000.0,200.0,2133.667611,32200000.0,287.14,1257.883494,52.557586,1.295794,831.623484,0,16.1,0.90,2.632463e+08
175996,Compressor,200.0,1050.0,2000000.0,200.0,2122.307093,32200000.0,287.14,1257.217458,52.838922,1.295997,829.434885,0,16.1,0.91,2.603890e+08
175997,Compressor,200.0,1050.0,2000000.0,200.0,2111.185650,32200000.0,287.14,1256.561094,53.117271,1.296197,827.286658,0,16.1,0.92,2.575933e+08
175998,Compressor,200.0,1050.0,2000000.0,200.0,2100.295757,32200000.0,287.14,1255.914262,53.392680,1.296395,825.177661,0,16.1,0.93,2.548573e+08


# ML

In [56]:
import os
import pickle
from colorama import Fore

import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error

In [57]:
MODELS = (
    LinearRegression,
    DecisionTreeRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor,
)
METRICS = (mean_absolute_error, mean_squared_error)

In [58]:
os.listdir("data")

['substance_parameters_13200.pkl.xz',
 'compressor_1000.pkl.xz',
 'substance_parameters_1000.pkl.xz']

In [59]:
db = load("data/compressor_1000.pkl.xz")

len(db)

176000

In [60]:
df = pd.DataFrame(db)
display(df)

,name,inlet_mass_flow,inlet_total_temperature,inlet_total_pressure,outlet_mass_flow,outlet_total_temperature,outlet_total_pressure,outlet_gas_const,outlet_heat_capacity_pressure,outlet_total_density,outlet_adiabatic_index,outlet_critical_sound_speed,leak,total_pressure_ratio,total_efficiency,power
0,Compressor,1.0,250.0,100000.0,1.0,258.243268,110000.0,287.14,1001.731420,1.483440,1.401824,294.206665,0,1.1,0.84,8.257334e+03
1,Compressor,1.0,250.0,100000.0,1.0,258.146291,110000.0,287.14,1001.730502,1.483997,1.401825,294.151441,0,1.1,0.85,8.160189e+03
2,Compressor,1.0,250.0,100000.0,1.0,258.051569,110000.0,287.14,1001.729617,1.484542,1.401825,294.097491,0,1.1,0.86,8.065303e+03
3,Compressor,1.0,250.0,100000.0,1.0,257.959024,110000.0,287.14,1001.728763,1.485075,1.401826,294.044772,0,1.1,0.87,7.972598e+03
4,Compressor,1.0,250.0,100000.0,1.0,257.868583,110000.0,287.14,1001.727939,1.485595,1.401826,293.993241,0,1.1,0.88,7.882001e+03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175995,Compressor,200.0,1050.0,2000000.0,200.0,2133.667611,32200000.0,287.14,1257.883494,52.557586,1.295794,831.623484,0,16.1,0.90,2.632463e+08
175996,Compressor,200.0,1050.0,2000000.0,200.0,2122.307093,32200000.0,287.14,1257.217458,52.838922,1.295997,829.434885,0,16.1,0.91,2.603890e+08
175997,Compressor,200.0,1050.0,2000000.0,200.0,2111.185650,32200000.0,287.14,1256.561094,53.117271,1.296197,827.286658,0,16.1,0.92,2.575933e+08
175998,Compressor,200.0,1050.0,2000000.0,200.0,2100.295757,32200000.0,287.14,1255.914262,53.392680,1.296395,825.177661,0,16.1,0.93,2.548573e+08


## Compressor

In [61]:
features = [f"inlet_{gtep.mf}", f"inlet_{gtep.TT}", f"inlet_{gtep.PP}", gtep.pipi, gtep.effeff, gtep.power]
targets = (gtep.pipi, gtep.effeff, gtep.power)

In [64]:
test_size = 0.25

models = []
for target in targets:
    x, y = df[features], df[target]
    x = x.drop([target], axis=1)
    print(f"{Fore.RED}{target=}{Fore.RESET} features={list(x.columns)}")

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=test_size, shuffle=True)

    """x_train = DictVectorizer(sparse=False).fit_transform(x_train.to_dict("records"))
    x_test = DictVectorizer(sparse=False).fit_transform(x_test.to_dict("records"))
    y_train = y_train.to_numpy()
    y_test = y_test.to_numpy()"""

    for Model in MODELS:
        model_name = Model.__name__
        print(f"\t{Fore.BLUE}{model_name}{Fore.RESET}")
        model = Model()
        model.fit(x_train, y_train)
        for metric in METRICS:
            tic = time.perf_counter()
            prediction = model.predict(x_test)
            tac = time.perf_counter()
            print(f"\t\t{Fore.MAGENTA}{tac - tic:.6f} s{Fore.RESET}")
            print(f"\t\t{Fore.YELLOW}{metric.__name__:<20}: {metric(y_test, prediction):.6f}{Fore.RESET}")

        dump(f"models/compressor_{target}_{model_name}.pkl.xz", model)

target='total_pressure_ratio' features=['inlet_mass_flow', 'inlet_total_temperature', 'inlet_total_pressure', 'total_efficiency', 'power']
	LinearRegression
		0.001624 s
		mean_absolute_error : 2.621228
		0.001037 s
		mean_squared_error  : 10.873151
	DecisionTreeRegressor
		0.004145 s
		mean_absolute_error : 0.000000
		0.003772 s
		mean_squared_error  : 0.000000
	RandomForestRegressor
		0.390543 s
		mean_absolute_error : 0.001637
		0.413039 s
		mean_squared_error  : 0.000052
	GradientBoostingRegressor
		0.038739 s
		mean_absolute_error : 1.601218
		0.039609 s
		mean_squared_error  : 3.743891
target='total_efficiency' features=['inlet_mass_flow', 'inlet_total_temperature', 'inlet_total_pressure', 'total_pressure_ratio', 'power']
	LinearRegression
		0.000640 s
		mean_absolute_error : 0.027227
		0.000210 s
		mean_squared_error  : 0.000991
	DecisionTreeRegressor
		0.006596 s
		mean_absolute_error : 0.000000
		0.006194 s
		mean_squared_error  : 0.000000
	RandomForestRegressor
		0.628902 s
	